In [ ]:
!pip install -q pdfplumber transformers faiss-cpu openai==0.28.0

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.30.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.


In [ ]:
import os
import pdfplumber
import numpy as np
import torch
import faiss
import textwrap
import openai



In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
pdf_files_path = '/content/drive/MyDrive/DL/RAG_data'

pdf_files = sorted([
    f for f in os.listdir(pdf_files_path)
    if f.lower().endswith('.pdf')
])

if not pdf_files:
    raise ValueError("No PDF files found.")

print("PDF files:", pdf_files)

PDF files: ['case-report-1.pdf', 'case-report-2.pdf', 'case-report-3.pdf']


Extract and chunk the PDFs

In [ ]:
def extract_text_from_pdf(pdf_path):
    all_text = []

    with pdfplumber.open(pdf_path) as pdf:
        for page_number, page in enumerate(pdf.pages, start=1):
            text = page.extract_text() or ""

            if text.strip():
                all_text.append({
                    "text": text,
                    "source": os.path.basename(pdf_path),
                    "page": page_number
                })

    return all_text


def create_chunks(pages, chunk_words=180, overlap=40):
    chunks = []
    step = chunk_words - overlap

    if step <= 0:
        raise ValueError("overlap must be smaller than chunk_words.")

    for page in pages:
        words = page["text"].split()

        for start in range(0, len(words), step):
            chunk_text = " ".join(
                words[start:start + chunk_words]
            )

            if chunk_text.strip():
                chunks.append({
                    "text": chunk_text,
                    "source": page["source"],
                    "page": page["page"]
                })

    return chunks


documents = []

for filename in pdf_files:
    path = os.path.join(pdf_files_path, filename)

    pages = extract_text_from_pdf(path)
    chunks = create_chunks(pages)

    documents.extend(chunks)

if not documents:
    raise ValueError(
        "No text extracted. Check whether the PDFs are scanned."
    )

print("Total chunks created:", len(documents))
print("\nExample chunk:\n", documents[0]["text"][:500])

Total chunks created: 78

Example chunk:
 CASE STUDY EXAMPLE INTRODUCTION Craniosacral Therapy is a gentle, hands-on form of manual therapy which addresses dysfunction within the craniosacral system, which follows the movement and flow of cerebrospinal fluid within the ventricles of the brain, the dural tube, and the central nervous system surrounding the spinal and cranial nerves. The therapist uses specific techniques to help restore physiological balance of the nervous system, the brain and cranial bones. Homeostasis is restored to t


Load DPR and encode the chunks

In [ ]:
from transformers import (
    DPRContextEncoder,
    DPRContextEncoderTokenizer,
    DPRQuestionEncoder,
    DPRQuestionEncoderTokenizer
)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

ctx_tokenizer = DPRContextEncoderTokenizer.from_pretrained(
    "facebook/dpr-ctx_encoder-single-nq-base"
)

ctx_model = DPRContextEncoder.from_pretrained(
    "facebook/dpr-ctx_encoder-single-nq-base"
).to(device)

q_tokenizer = DPRQuestionEncoderTokenizer.from_pretrained(
    "facebook/dpr-question_encoder-single-nq-base"
)

q_model = DPRQuestionEncoder.from_pretrained(
    "facebook/dpr-question_encoder-single-nq-base"
).to(device)

ctx_model.eval()
q_model.eval()


def encode_documents(documents, batch_size=8):
    embeddings = []

    for start in range(0, len(documents), batch_size):
        texts = [
            doc["text"]
            for doc in documents[start:start + batch_size]
        ]

        encoded = ctx_tokenizer(
            texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=256
        )

        encoded = {
            key: value.to(device)
            for key, value in encoded.items()
        }

        with torch.no_grad():
            output = ctx_model(**encoded).pooler_output

            output = torch.nn.functional.normalize(
                output, p=2, dim=1
            )

        embeddings.append(
            output.cpu().numpy().astype("float32")
        )

    return np.concatenate(embeddings, axis=0)


doc_embeddings = encode_documents(documents)

print("Embedding matrix:", doc_embeddings.shape)

tokenizer_config.json:   0%|          | 0.00/28.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/492 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  438MB            

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  438MB            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

model.safetensors: downloading bytes:           |  0.00B            

[transformers] DPRQuestionEncoder LOAD REPORT from: facebook/dpr-question_encoder-single-nq-base
Key                                             | Status     |  | 
------------------------------------------------+------------+--+-
question_encoder.bert_model.pooler.dense.bias   | UNEXPECTED |  | 
question_encoder.bert_model.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding matrix: (78, 768)


Create the FAISS index

In [ ]:
dimension = doc_embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)
index.add(doc_embeddings)

print("Indexed chunks:", index.ntotal)

Indexed chunks: 78


Retrieve the top-k chunks

In [ ]:
def search(query, k=4):
    encoded_query = q_tokenizer(
        query,
        return_tensors="pt",
        truncation=True,
        max_length=128
    )

    encoded_query = {
        key: value.to(device)
        for key, value in encoded_query.items()
    }

    with torch.no_grad():
        query_embedding = q_model(
            **encoded_query
        ).pooler_output

        query_embedding = torch.nn.functional.normalize(
            query_embedding, p=2, dim=1
        )

    query_vector = (
        query_embedding.cpu().numpy().astype("float32")
    )

    k = min(k, index.ntotal)
    scores, indices = index.search(query_vector, k)

    retrieved_docs = []

    for score, idx in zip(scores[0], indices[0]):
        if idx < 0:
            continue

        doc = documents[int(idx)]

        retrieved_docs.append({
            "text": doc["text"],
            "source": doc["source"],
            "page": doc["page"],
            "score": float(score)
        })

    return retrieved_docs

In [ ]:
from getpass import getpass

api_key = getpass("Enter your OpenAI API key: ")

openai.api_key = api_key

Enter your OpenAI API key: ··········


In [ ]:
def generate_answer(query, retrieved_docs):
    if not retrieved_docs:
        return "No relevant passages were retrieved."

    context = "\n\n".join([
        (
            f"[Source: {doc['source']}, "
            f"Page: {doc['page']}]\n"
            f"{doc['text']}"
        )
        for doc in retrieved_docs
    ])

    system_prompt = """
You are a document-grounded question-answering assistant.

Answer using the supplied document passages.
Do not invent information that is not supported by the context.
If the context is insufficient, clearly state that.
Cite source filenames and page numbers where appropriate.
For medical questions, provide general information and do not
present the answer as a diagnosis.
"""

    user_prompt = f"""
Question:
{query}

Retrieved document passages:
{context}

Provide a clear, relevant answer based on the passages.
"""

    response = openai.ChatCompletion.create(
    model="gpt-5.6-sol",
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ],
    max_completion_tokens=500)

    return response["choices"][0]["message"]["content"].strip()

In [ ]:
query = (
    "What is lightheadedness and how is it related "
    "to rheumatoid arthritis?"
)

retrieved_docs = search(query, k=4)

print("Retrieved passages:")

for i, doc in enumerate(retrieved_docs, start=1):
    print(
        f"\n{i}. {doc['source']} | Page {doc['page']} "
        f"| Similarity: {doc['score']:.4f}"
    )
    print(doc["text"][:300])

answer = generate_answer(query, retrieved_docs)

print("\n" + "=" * 70)
print("RAG ANSWER")
print("=" * 70)
print(textwrap.fill(answer, width=80))

Retrieved passages:

1. case-report-2.pdf | Page 1 | Similarity: 0.6425
and muscle weakness. Patient began experiencing symptoms 4 months ago (November 2017). At that time he experienced fatigue and joint pain in the knees and hands. He was diagnosed with Rheumatoid Arthritis. He was given a short course of corticosteroids at that time that alleviated his symptoms. He w

2. case-report-1.pdf | Page 6 | Similarity: 0.6274
6 5 4 3 2 1 0 Dorsiflexion Plantarflexion Inversion Eversion Fig. 2 Standard Manual Muscle Testing pre- (blue) and post-craniosacral therapy (red) Subjectively, the patient reported a 0 on the pain scale of 0-10 in all of her ADL’s, including standing for extended periods of time, stair climbing, ru

3. case-report-2.pdf | Page 4 | Similarity: 0.6165
night sweats alongside symptoms of burning eyes and dry mouth, decreased RBC and Hemoglobin, and a CT with an enlarged parotid gland.

4. case-report-2.pdf | Page 1 | Similarity: 0.6158
in frequency and duration and over t